# MambaIR — Light SR Eğitimi (Colab)
## 🔍 GPU Tespiti — İlk Bu Hücreyi Çalıştır

In [ ]:
# GPU0 — GPU Tespiti
import torch, subprocess
if not torch.cuda.is_available():
    print("❌ GPU bulunamadı!")
else:
    cc = torch.cuda.get_device_capability()
    sm = cc[0] * 10 + cc[1]
    print(f"GPU         : {torch.cuda.get_device_name(0)} (SM{sm})")
    print(f"PyTorch     : {torch.__version__}")
    print(f"CUDA built  : {torch.version.cuda}")
    if sm >= 120: print("⚠️  Blackwell → A1b")
    elif sm >= 89: print("⚠️  Ada Lovelace → A1a (cu124)")
    else: print("✅ Turing/Ampere → A1a (cu118)")


In [ ]:
# GPU_BENCH — Anlık GPU izleme (E1 çalışırken başka sekmede çalıştır)
import subprocess, time
for _ in range(5):  # 5 kez ölç
    r = subprocess.run(
        ["nvidia-smi","--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True)
    name, g, m, mu, mt, temp = r.stdout.strip().split(", ")
    print(f"GPU {g:>3}% compute | MEM {mu:>5}/{mt} MB ({m:>3}%) | 🌡{temp}°C")
    time.sleep(3)


## 🔧 BÖLÜM A — Ortam Kurulumu

In [ ]:
# A1 — PyTorch + Mamba kurulumu (tekrar çalıştırılabilir)
import importlib

def _ok(pkg):
    try: importlib.import_module(pkg); return True
    except: return False

import torch
already_ok = _ok("causal_conv1d") and "2.1.1" in torch.__version__

if already_ok:
    print("✅ Zaten kurulu — A2'ye geçin")
else:
    import subprocess
    print("📦 torch 2.1.1 kuruluyor...")
    subprocess.run(["pip", "install", "-q", "torch==2.1.1", "torchvision==0.16.1",
                    "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
    subprocess.run(["pip", "install", "-q", "numpy<2.0"], check=True)
    print("📦 causal_conv1d 1.1.1 kuruluyor...")
    subprocess.run(["pip", "install", "-q", "causal_conv1d==1.1.1", "--no-build-isolation"], check=True)
    print("📦 mamba-ssm 1.2.0.post1 kuruluyor...")
    subprocess.run(["pip", "install", "-q", "mamba-ssm==1.2.0.post1", "--no-build-isolation"], check=True)
    print("✅ Kurulum tamam!")
    print("⚠️  Şimdi A2 hücresini çalıştırıp Runtime'ı restart edin.")


In [ ]:
# A2 — Runtime Restart
import os
os.kill(os.getpid(), 9)


In [ ]:
# A3 — Ortam Doğrulama (restart sonrası, salt okunur)
import numpy as np, torch, causal_conv1d, mamba_ssm

print(f"NumPy         : {np.__version__}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA          : {torch.version.cuda}")
print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"causal_conv1d : {causal_conv1d.__version__}")
print(f"mamba_ssm     : {mamba_ssm.__version__}")
print("\n✅ Mamba ortamı hazır")


## 📦 BÖLÜM B — Repo & Bağımlılıklar

In [ ]:
# B1 — Repo Clone (zaten varsa atlar)
import os

REPO_PATH = "/content/MambaIR"

if os.path.exists(REPO_PATH):
    print(f"✅ Repo zaten mevcut: {REPO_PATH}")
else:
    !git clone https://github.com/alperslmz/MambaIR-AS.git {REPO_PATH}
    print("✅ Repo clone edildi")


In [ ]:
# B2 — Bağımlılıklar
!pip install -q timm einops lmdb tb-nightly
print("✅ timm, einops, lmdb, tb-nightly")


In [ ]:
# B3 — basicsr Path
import sys, subprocess; REPO_PATH="/content/MambaIR"
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH); print("✅ sys.path'e eklendi")
else: print("✅ sys.path'de zaten var")a
r = subprocess.run(["python","setup.py","develop","--no-deps"],capture_output=True,text=True,cwd=REPO_PATH)
print("✅ setup.py OK" if r.returncode==0 else "⚠️  sys.path yeterli, devam")


In [ ]:
# B4 — basicsr Import Testi (salt okunur)
import importlib
importlib.invalidate_caches()

import basicsr
from basicsr.models import build_model
from basicsr.data import build_dataset

print(f"✅ basicsr   : {basicsr.__file__}")
print("✅ build_model OK")
print("✅ build_dataset OK")
print("\n🚀 basicsr hazır")


## 💾 BÖLÜM C — Dataset

In [ ]:
# C1 — Drive Mount (zaten bağlıysa atlar)
from google.colab import drive
import os

if os.path.exists("/content/drive/MyDrive"):
    print("✅ Drive zaten bağlı")
else:
    drive.mount("/content/drive")
    print("✅ Drive bağlandı")


In [ ]:
# C2 — Dataset Zip Açma (zaten açıksa atlar)
import zipfile, os

ZIP_PATH    = "/content/drive/MyDrive/IRdataset/IRdataset.zip"           # <-- DEĞİŞTİR
EXTRACT_TO  = "/content/MambaIR/datasets/"
MARKER_FILE = "/content/MambaIR/datasets/IRdataset/.extracted" # tamamlandı işareti

if os.path.exists(MARKER_FILE):
    print("✅ Dataset zaten açık, atlanıyor")
else:
    os.makedirs(EXTRACT_TO, exist_ok=True)
    print(f"📦 Zip açılıyor: {ZIP_PATH}")
    print("   (Büyük dataset için 3-10 dakika sürebilir...)")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_TO)
    # Tamamlandı işareti bırak
    open(MARKER_FILE, "w").close()
    print("✅ Dataset açıldı")

# İçerik kontrolü
base = "/content/MambaIR/datasets/IRdataset"
checks = [
    f"{base}/DIV2K/DIV2K_train_HR",
    f"{base}/DIV2K/DIV2K_train_LR_bicubic/X2",
    f"{base}/SR/Urban100/HR",
]
print("\n── Kritik Klasör Kontrolü ──")
for p in checks:
    print(f"  {'✅' if os.path.exists(p) else '❌'} {p}")


## ⚙️ BÖLÜM D — Config

In [ ]:
# D1 — Config YAML Oluştur
import yaml, os, sys, torch
REPO_PATH = "/content/MambaIR"
if REPO_PATH not in sys.path: sys.path.insert(0, REPO_PATH)

# ══════════════════════════════════════════════
TEST_MODE = False  # True: quick test | False: real train
SCALE     = 2      # 2 / 3 / 4
USE_AMP   = False  
# ══════════════════════════════════════════════

if TEST_MODE:
    TOTAL_ITER=200; BATCH_PER_GPU=4; NUM_WORKERS=4; ENLARGE_RATIO=1
    CHECKPOINT_FREQ=100; VAL_FREQ=100; PRINT_FREQ=10; config_suffix="_TEST"
else:

    TOTAL_ITER=75000; BATCH_PER_GPU=8; NUM_WORKERS=4; ENLARGE_RATIO=75

    CHECKPOINT_FREQ=7500; VAL_FREQ=7500; PRINT_FREQ=200; config_suffix=""

BASE="/content/MambaIR/datasets/IRdataset"; DRIVE_OUT="/content/drive/MyDrive/MambaIR_experiments"
LR_DIR_MAP={2:"DIV2K_train_LR_bicubic/X2",3:"DIV2K_train_LR_bicubic-3/X3",4:"DIV2K_train_LR_bicubic-2/X4"}
GT_TRAIN=f"{BASE}/DIV2K/DIV2K_train_HR"; LR_TRAIN=f"{BASE}/DIV2K/{LR_DIR_MAP[SCALE]}"
GT_VAL=f"{BASE}/SR/Urban100/HR"; LR_VAL=f"{BASE}/SR/Urban100/LR_bicubic/X{SCALE}"
TMPL=f"{{}}x{SCALE}"; config_name=f"MambaIR_lightSR_x{SCALE}_colab{config_suffix}"
CONFIG_PATH=f"{REPO_PATH}/options/train/{config_name}.yml"

cfg={
    "name":config_name,"model_type":"MambaIRModel","scale":SCALE,"num_gpu":1,"manual_seed":10,
    "datasets":{"train":{"task":"SR","name":"DIV2K","type":"PairedImageDataset",
        "dataroot_gt":[GT_TRAIN],"dataroot_lq":[LR_TRAIN],"filename_tmpl":TMPL,
        "io_backend":{"type":"disk"},"gt_size":128 if SCALE==2 else 192,
        "use_hflip":True,"use_rot":True,"use_shuffle":True,
        "num_worker_per_gpu":NUM_WORKERS,"batch_size_per_gpu":BATCH_PER_GPU,
        "dataset_enlarge_ratio":ENLARGE_RATIO,"prefetch_mode":None},
        "val":{"name":"Urban100","type":"PairedImageDataset",
        "dataroot_gt":GT_VAL,"dataroot_lq":LR_VAL,"filename_tmpl":TMPL,"io_backend":{"type":"disk"}}},
    "network_g":{"type":"MambaIR","upscale":SCALE,"in_chans":3,"img_size":64,
        "img_range":1.0,"d_state":10,"depths":[6,6,6,6],"embed_dim":60,"mlp_ratio":1.2,
        "upsampler":"pixelshuffledirect","resi_connection":"1conv"},
    "path":{"pretrain_network_g":None,"strict_load_g":True,
        "resume_state":None,"experiments_root":DRIVE_OUT},
    "train":{
        "optim_g":{"type":"Adam","lr":2e-4,"weight_decay":0,"betas":[0.9,0.99]},
        "scheduler":{"type":"CosineAnnealingRestartLR",
        "periods":[100,100] if TEST_MODE else [37500, 37500],"restart_weights":[1,1],"eta_min":1e-7},
        "total_iter":TOTAL_ITER,"warmup_iter":-1,
        "pixel_opt":{"type":"L1Loss","loss_weight":1.0,"reduction":"mean"},
        "use_amp":USE_AMP},
    "val":{"val_freq":VAL_FREQ,"save_img":False,
        "metrics":{"psnr":{"type":"calculate_psnr","crop_border":SCALE,"test_y_channel":True}}},
    "logger":{"print_freq":PRINT_FREQ,"save_checkpoint_freq":CHECKPOINT_FREQ,
        "use_tb_logger":True,"wandb":{"project":None,"resume_id":None}},
    "dist_params":{"backend":"nccl","port":29500}}

os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
with open(CONFIG_PATH,"w") as f: yaml.dump(cfg,f,default_flow_style=False,allow_unicode=True)

mode_str="🧪 TEST" if TEST_MODE else "🏋️ 10 SAATLİK OPTİMİZE EĞİTİM"
amp_str="ON ⚡" if USE_AMP else "OFF"
print(f"{mode_str} | scale=x{SCALE} | iter={TOTAL_ITER} | AMP={amp_str}")
print(f"batch={BATCH_PER_GPU} | workers={NUM_WORKERS} | enlarge={ENLARGE_RATIO}")
print(f"Config: {CONFIG_PATH}")
all_ok=True; print("\n── Path Kontrolü ──")
for label,path in [("GT train",GT_TRAIN),("LQ train",LR_TRAIN),("GT val",GT_VAL),("LQ val",LR_VAL)]:
    ok=os.path.exists(path); all_ok=all_ok and ok
    print(f"  {'✅' if ok else '❌'} {label}: {path}")
print("\n✅ E1 ile eğitimi başlatın" if all_ok else "\n❌ Eksik path — C2")


## 🚀 BÖLÜM E — Eğitim

In [ ]:
# E1 — Eğitimi Başlat
import os
TEST_MODE = False  # D1 ile aynı
SCALE     = 2      # D1 ile aynı
config_suffix = "_TEST" if TEST_MODE else ""
OPT = f"options/train/MambaIR_lightSR_x{SCALE}_colab{config_suffix}.yml"
os.chdir("/content/MambaIR")

!sed -i "s|opt\['root_path'\], 'tb_logger'|opt\['path'\]\['experiments_root'\], 'tb_logger'|g" basicsr/train.py

print(f"Config: {OPT}\n🏋️  Başlıyor...\n")
!PYTHONPATH=/content/MambaIR python basicsr/train.py -opt {OPT} --launcher none


In [ ]:
# E2 — Son Checkpoint'i Config'e Yaz
import yaml, glob, os
TEST_MODE = False; SCALE = 2
config_suffix = "_TEST" if TEST_MODE else ""
config_name = f"MambaIR_lightSR_x{SCALE}_colab{config_suffix}"
CONFIG_PATH = f"/content/MambaIR/options/train/{config_name}.yml"
state_dir = f"/content/drive/MyDrive/MambaIR_experiments/{config_name}/training_states"
states = glob.glob(f"{state_dir}/*.state")

if not states:
    print(f"⚠️  State yok: {state_dir}")
else:
    states.sort(key=lambda x: int(os.path.basename(x).replace('.state', '')))
    last=states[-1]; last_iter=os.path.basename(last).replace(".state","")
    print(f"🔄 Son checkpoint: iter {last_iter} ({len(states)} state)")
    with open(CONFIG_PATH,"r") as f: cfg=yaml.safe_load(f)
    cfg["path"]["resume_state"]=last
    with open(CONFIG_PATH,"w") as f: yaml.dump(cfg,f,default_flow_style=False,allow_unicode=True)
    print("✅ Config güncellendi → E1'i çalıştır")


## 📊 BÖLÜM F — İzleme

In [ ]:
# F1 — TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/MambaIR_experiments/tb_logger


In [ ]:
# F2 — Checkpoint Durumu
import glob, os
TEST_MODE=False; SCALE=2
config_name=f"MambaIR_lightSR_x{SCALE}_colab{'_TEST' if TEST_MODE else ''}"
ckpt_dir=f"/content/drive/MyDrive/MambaIR_experiments/{config_name}"
models=glob.glob(f"{ckpt_dir}/models/*.pth")
states=glob.glob(f"{ckpt_dir}/training_states/*.state")

def get_num(p): 
    try: 
        return int(os.path.basename(p).split('.')[0].split('_')[-1])
    except: 
        return 0
models.sort(key=get_num)
states.sort(key=get_num)

print(f"📁 {config_name}")
for m in models: print(f"   model: {os.path.basename(m)} ({os.path.getsize(m)/1e6:.1f} MB)")
for s in states: print(f"   state: iter {os.path.basename(s).replace('.state','')}")
if states: print(f"\n✅ Son: iter {os.path.basename(states[-1]).replace('.state','')}")
else: print("\n⚠️  Henüz checkpoint yok")
